In [1]:
# importando os scripts de consulta

import sys
import os

# Altera o caminho para a pasta raiz do projeto (um nível acima)
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Adiciona a raiz do projeto ao caminho de busca do Python se ainda não estiver lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

    
import pandas as pd
from scripts.crm_avaliacoes import load_avaliacoes
from scripts.crm_leads_escolas_parceiras import load_leads

In [2]:
avaliacoes = load_avaliacoes()

Página 1: 38 registros


In [3]:
avaliacoes.head(2)

,cad_avaliacaoid,createdon,cad_fatoravaliacaoid,cad_inscricaocandidatoid,cad_situacaoavaliacao,cad_valor
0,{795995C3-48B8-F111-AAAE-000D3AC0F764},NaT,Critério Nota Final Online Agendado (Calculado),CAN-007212-TRI,Não Avaliado,0
1,{7A5995C3-48B8-F111-AAAE-000D3AC0F764},NaT,Taxa de Matricula Vestibular Agendado,CAN-007212-TRI,Não Avaliado,0


In [4]:
avaliacoes['cad_valor'] = avaliacoes['cad_valor'].astype(float)
avaliacoes_agrupado = avaliacoes[[ 'cad_inscricaocandidatoid' , 'cad_fatoravaliacaoid', 'cad_valor']].groupby(['cad_inscricaocandidatoid', 'cad_fatoravaliacaoid']).agg({'cad_valor': 'sum'}).reset_index()

In [5]:
# construindo a tabela unstack

avaliacoes_pivot = avaliacoes_agrupado.pivot(index='cad_inscricaocandidatoid', columns='cad_fatoravaliacaoid', values='cad_valor').fillna(0)

In [6]:
colunas = ["Fator Prova Matemática Escolas (Agendado)", "Fator Prova Português Escolas (Agendado)",
           "Fator Redação Online Escolas"]

In [7]:
avaliacoes_pivot = avaliacoes_pivot[colunas]

# Somar os valores das colunas selecionadas e criar uma nova coluna chamada 'Total'
avaliacoes_pivot['Nota final'] = avaliacoes_pivot[colunas].sum(axis=1)
avaliacoes_pivot

cad_fatoravaliacaoid,Fator Prova Matemática Escolas (Agendado),Fator Prova Português Escolas (Agendado),Fator Redação Online Escolas,Nota final
cad_inscricaocandidatoid,,,,
CAN-007209-TRI,0.0,0.0,0.0,0.0
CAN-007212-TRI,0.0,0.0,0.0,0.0
CAN-007213-TRI,0.0,0.0,0.0,0.0
CAN-007247-TRI,0.0,0.0,0.0,0.0
CAN-007253-TRI,0.0,0.0,0.0,0.0
CAN-007257-TRI,0.0,0.0,0.0,0.0
CAN-007265-TRI,0.0,0.0,0.0,0.0
CAN-007266-TRI,35.0,20.0,50.0,105.0


### Carregamento dos leads

In [57]:
leads = load_leads()

Página 1: 8 registros


In [58]:
leads.columns

Index(['Escola Lorena', 'Inscrição Atual', 'cpf', 'Tabulação Realizada',
       'Nome Completo', 'Nome', 'Sobrenome', 'Email', 'Celular', 'ID Lead',
       'Data Criação', 'Idade', 'Concurso Atual', 'Concurso de Origem',
       'Data Nascimento', 'Oferta Atual', 'Situação Funil', 'Unidade Lead',
       'Linha de Negócio', 'Proprietário'],
      dtype='object')

In [59]:
colunas = ['ID Lead', 'Nome Completo', 'Inscrição Atual', 'Concurso Atual', 'Escola Lorena', 'Idade']

leads = leads[colunas]

In [60]:
leads['Idade'].describe()

count      8
unique     1
top       27
freq       8
Name: Idade, dtype: object

In [11]:
# juntando as duas informações


df = pd.merge(leads, avaliacoes_pivot, how='left', left_on='Inscrição Atual', right_on='cad_inscricaocandidatoid')

In [12]:
df

,ID Lead,Nome Completo,Inscrição Atual,Concurso Atual,Escola Lorena,Idade,Fator Prova Matemática Escolas (Agendado),Fator Prova Português Escolas (Agendado),Fator Redação Online Escolas,Nota final
0,{4C17FB54-4387-424C-BEFD-116174126C4C},Hugo Teste,CAN-007266-TRI,Vestibular Escolas Parceiras Lorena 27.1,Barão Da Bocaina - Areias,27,35.0,20.0,50.0,105.0
1,{CFD10BFF-D972-44FA-873A-14C956617867},Arthur Teste,CAN-007265-TRI,Vestibular Escolas Parceiras Lorena 27.1,Centro Educacional Tiradentes - Volta Redonda,27,0.0,0.0,0.0,0.0
2,{D19BDE38-79D3-4451-A910-DC010DBEBBD0},Joao Teste Prova,CAN-007257-TRI,Vestibular Escolas Parceiras Lorena 27.1,Canção Nova - Cachoeira Paulista,27,0.0,0.0,0.0,0.0
3,{3DBDBFCC-0BA1-40BB-A126-D52D582FF748},Hugo Prova Teste,CAN-007253-TRI,Vestibular Escolas Parceiras Lorena 27.1,Canção Nova - Cachoeira Paulista,27,0.0,0.0,0.0,0.0
4,{BE2441D9-182F-495A-96A8-601501F22D15},Hugo Prova Teste,CAN-007247-TRI,Vestibular Escolas Parceiras Lorena 27.1,Centro Educacional Aquarela - Itanhandu,27,0.0,0.0,0.0,0.0
5,{81E2F059-94B1-45A8-9E1C-E77C05ECE33C},Prova Tres Teste,CAN-007213-TRI,Vestibular Escolas Parceiras Lorena 27.1,Centro Educacional Aquarela - Itanhandu,27,0.0,0.0,0.0,0.0
6,{4D788A87-D0A7-41BC-BB1C-7A4DB66A3CED},Prova Teste Dois,CAN-007212-TRI,Vestibular Escolas Parceiras Lorena 27.1,Canção Nova - Cachoeira Paulista,27,0.0,0.0,0.0,0.0
7,{F0979084-0059-4DB2-B52D-4A598828C17F},Prova Um Teste,CAN-007209-TRI,Vestibular Escolas Parceiras Lorena 27.1,Basic Taubaté - Taubaté,27,0.0,0.0,0.0,0.0


In [13]:
'''Critérios de classificação dos candidatos:

Caso 1
Caso sua escola tenha 10 ou mais candidatos,  o candidato é rankeado dentro de sua escola


Caso 2
Caso sua escola menos que  10 candidatos,  o candidato é rankeado como Geral

Critérios:
- maior nota final
- maior nota de redação
- maior idade


'''

'Critérios de classificação dos candidatos:\n\nCaso 1\nCaso sua escola tenha 10 ou mais candidatos,  o candidato é rankeado dentro de sua escola\n\n\nCaso 2\nCaso sua escola menos que  10 candidatos,  o candidato é rankeado como Geral\n\nCritérios:\n- maior nota final\n- maior nota de redação\n- maior idade\n\n\n'

In [14]:
# Aplica os critérios de classificação e salva um resultado auditável
coluna_redacao = 'Fator Redação Online Escolas'
colunas_notas = [
    'Fator Prova Matemática Escolas (Agendado)',
    'Fator Prova Português Escolas (Agendado)',
    coluna_redacao,
    'Nota final',
    'Idade',
]

df_classificacao = df.copy()
df_classificacao[colunas_notas] = df_classificacao[colunas_notas].apply(
    pd.to_numeric, errors='coerce'
)
df_classificacao[colunas_notas] = df_classificacao[colunas_notas].fillna(0)

df_classificacao['Quantidade na escola'] = (
    df_classificacao.groupby('Escola Lorena')['ID Lead'].transform('size')
)
df_classificacao['Grupo de classificação'] = df_classificacao['Escola Lorena'].where(
    df_classificacao['Quantidade na escola'].ge(10),
    'Geral',
)

df_classificacao = df_classificacao.sort_values(
    by=[
        'Grupo de classificação',
        'Nota final',
        coluna_redacao,
        'Idade',
        'Nome Completo',
        'ID Lead',
    ],
    ascending=[True, False, False, False, True, True],
    kind='stable',
).reset_index(drop=True)

df_classificacao['Classificação'] = (
    df_classificacao.groupby('Grupo de classificação').cumcount().add(1)
)

colunas_resultado = [
    'Classificação',
    'Grupo de classificação',
    'Quantidade na escola',
    'Escola Lorena',
    'Nome Completo',
    'Inscrição Atual',
    'Nota final',
    coluna_redacao,
    'Idade',
    'ID Lead',
]
resultado_classificacao = df_classificacao[colunas_resultado]
resultado_classificacao.to_csv(
    'classificacao_candidatos.csv',
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

resultado_classificacao


,Classificação,Grupo de classificação,Quantidade na escola,Escola Lorena,Nome Completo,Inscrição Atual,Nota final,Fator Redação Online Escolas,Idade,ID Lead
0,1,Geral,1,Barão Da Bocaina - Areias,Hugo Teste,CAN-007266-TRI,105.0,50.0,27,{4C17FB54-4387-424C-BEFD-116174126C4C}
1,2,Geral,1,Centro Educacional Tiradentes - Volta Redonda,Arthur Teste,CAN-007265-TRI,0.0,0.0,27,{CFD10BFF-D972-44FA-873A-14C956617867}
2,3,Geral,3,Canção Nova - Cachoeira Paulista,Hugo Prova Teste,CAN-007253-TRI,0.0,0.0,27,{3DBDBFCC-0BA1-40BB-A126-D52D582FF748}
3,4,Geral,2,Centro Educacional Aquarela - Itanhandu,Hugo Prova Teste,CAN-007247-TRI,0.0,0.0,27,{BE2441D9-182F-495A-96A8-601501F22D15}
4,5,Geral,3,Canção Nova - Cachoeira Paulista,Joao Teste Prova,CAN-007257-TRI,0.0,0.0,27,{D19BDE38-79D3-4451-A910-DC010DBEBBD0}
5,6,Geral,3,Canção Nova - Cachoeira Paulista,Prova Teste Dois,CAN-007212-TRI,0.0,0.0,27,{4D788A87-D0A7-41BC-BB1C-7A4DB66A3CED}
6,7,Geral,2,Centro Educacional Aquarela - Itanhandu,Prova Tres Teste,CAN-007213-TRI,0.0,0.0,27,{81E2F059-94B1-45A8-9E1C-E77C05ECE33C}
7,8,Geral,1,Basic Taubaté - Taubaté,Prova Um Teste,CAN-007209-TRI,0.0,0.0,27,{F0979084-0059-4DB2-B52D-4A598828C17F}
